#📹ℹ️✍️📝 Extraer el texto de Tik Tok

##0. Parámetros generales

In [1]:
TIKTOK_URL = "https://vt.tiktok.com/ZSavfXXUo/"  # cambia aquí
OUT_DIR = "/content/tiktok_union"
LANG = "es"
WHISPER_MODEL = "small"   # "base" (más rápido) / "medium" (más precisión, más lento)
GENERAR_SRT = True

##1 — Instalación (ffmpeg + dependencias)

In [2]:
!apt-get -y update

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.8 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,891 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,701 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,677 kB]
Get:14 https:/

In [3]:
!apt-get -y install ffmpeg

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.


In [4]:
# OJO: en Colab evita actualizar requests. Solo instala lo que necesitas.
!pip -q install -U yt-dlp openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.0/182.0 kB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 35.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.0 MB/s eta 0:00:00


##2 — Imports + Helpers base

In [5]:
import os, json, re, glob, subprocess
import requests
import yt_dlp
import whisper

In [6]:
os.makedirs(OUT_DIR, exist_ok=True)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "es-CL,es;q=0.9,en;q=0.8",
    "Referer": "https://www.tiktok.com/",
}

YDL_OPTS_INFO = {
    "quiet": True,
    "no_warnings": True,
    "noplaylist": True,
}

def pick(*vals):
    for v in vals:
        if v not in (None, "", "NA"):
            return v
    return None

def get_final_url(url: str) -> str | None:
    try:
        r = requests.get(url, headers=HEADERS, allow_redirects=True, timeout=20)
        return r.url
    except Exception:
        return None

def infer_uploader_from_url(url: str | None) -> str | None:
    if not url:
        return None
    m = re.search(r"tiktok\.com/@([^/]+)/", url)
    return m.group(1) if m else None

def fetch_html(url: str | None) -> str | None:
    if not url:
        return None
    try:
        r = requests.get(url, headers=HEADERS, timeout=20)
        if r.status_code != 200:
            return None
        return r.text
    except Exception:
        return None

def guess_music_from_html(html: str | None):
    if not html:
        return None, None
    patterns = [
        (r'"musicName"\s*:\s*"([^"]+)"', None),
        (r'"soundTitle"\s*:\s*"([^"]+)"', None),
        (r'"title"\s*:\s*"([^"]+)"\s*,\s*"authorName"\s*:\s*"([^"]+)"', "pair"),
    ]
    for pat, mode in patterns:
        m = re.search(pat, html)
        if not m:
            continue
        if mode == "pair" and m.lastindex and m.lastindex >= 2:
            return m.group(1), m.group(2)
        return m.group(1), None
    return None, None


##3 — Función 1: obtener_metadata(url)

Devuelve exactamente el diccionario que pediste + guarda video_info_raw.json y metadata.json.

In [7]:
def obtener_metadata(tiktok_url: str, out_dir: str) -> dict:
    os.makedirs(out_dir, exist_ok=True)

    final_url = get_final_url(tiktok_url)
    is_photo = bool(final_url and "/photo/" in final_url)
    uploader_guess = infer_uploader_from_url(final_url)

    # 1) intentar info con yt-dlp
    info = None
    status = "ok"
    error_msg = None

    try:
        with yt_dlp.YoutubeDL(YDL_OPTS_INFO) as ydl:
            info = ydl.extract_info(tiktok_url, download=False)
    except Exception as e:
        status = "error"
        error_msg = str(e)
        info = None

    # 2) fallback HTML para track/artist (best-effort)
    html = fetch_html(final_url) if (is_photo or info is None) else None
    track_guess, artist_guess = guess_music_from_html(html)

    # 3) raw unificado
    raw = info if info is not None else {}
    raw.update({
        "status": "ok" if info is not None else status,
        "error": None if info is not None else error_msg,
        "input_url": tiktok_url,
        "final_url": final_url,
        "content_type_guess": "photo" if is_photo else "video/unknown",
        "uploader_guess": uploader_guess,
        "track_guess": track_guess,
        "artist_guess": artist_guess,
    })

    raw_path = f"{out_dir}/video_info_raw.json"
    with open(raw_path, "w", encoding="utf-8") as f:
        json.dump(raw, f, ensure_ascii=False, indent=2)

    # 4) meta final (lo que tú necesitas)
    uploader_name = pick(raw.get("uploader"), raw.get("channel"), raw.get("creator"), raw.get("uploader_guess"))
    uploader_id   = pick(raw.get("uploader_id"), raw.get("channel_id"))
    uploader_url  = pick(raw.get("uploader_url"), raw.get("channel_url"))

    track  = pick(raw.get("track"), raw.get("alt_title"), raw.get("track_guess"))
    artist = pick(raw.get("artist"), raw.get("composer"), raw.get("artist_guess"))
    has_music = bool(track or artist)

    content_type = "photo" if (raw.get("content_type_guess") == "photo" or (final_url and "/photo/" in final_url)) else "video/unknown"

    meta = {
        "tiktok_url": tiktok_url,
        "final_url": final_url,
        "content_type": content_type,

        "uploader_name": uploader_name,
        "uploader_id": uploader_id,
        "uploader_url": uploader_url,

        "has_music": has_music,
        "track": track,
        "artist": artist,

        "id": raw.get("id"),
        "title": raw.get("title"),
        "duration_sec": raw.get("duration"),
        "view_count": raw.get("view_count"),
        "like_count": raw.get("like_count"),
        "comment_count": raw.get("comment_count"),
        "repost_count": raw.get("repost_count"),

        "status": raw.get("status"),
        "error": raw.get("error"),
    }

    meta_path = f"{out_dir}/metadata.json"
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    return meta


##4 — Funciones 2 y 3: descarga + obtener_texto_whisper(...)

*   Descarga video con yt-dlp (solo si no es photo)
*   Extrae audio wav mono 16k
*   Transcribe con Whisper
*   Guarda transcripcion.txt, segments.json
*   Opcional: .srt

In [8]:
def descargar_video(tiktok_url: str, out_dir: str) -> str:
    """
    Descarga video a out_dir/tiktok.(ext) y retorna el path real del video.
    """
    os.makedirs(out_dir, exist_ok=True)
    out_tpl = f"{out_dir}/tiktok.%(ext)s"
    cmd = f'''yt-dlp -o "{out_tpl}" "{tiktok_url}"'''
    subprocess.run(cmd, shell=True, check=True)

    videos = glob.glob(f"{out_dir}/*.mp4") + glob.glob(f"{out_dir}/*.webm") + glob.glob(f"{out_dir}/*.mkv")
    if not videos:
        raise FileNotFoundError("No se descargó ningún video (mp4/webm/mkv).")
    return videos[0]

def extraer_audio(video_path: str, out_dir: str) -> str:
    audio_path = f"{out_dir}/audio.wav"
    cmd = f'''ffmpeg -y -i "{video_path}" -ac 1 -ar 16000 "{audio_path}"'''
    subprocess.run(cmd, shell=True, check=True)
    return audio_path

def obtener_texto_whisper(audio_path: str, out_dir: str, language: str = "es", model_name: str = "small", generar_srt: bool = True) -> dict:
    """
    Transcribe audio y guarda:
      - transcripcion.txt
      - segments.json
      - (opcional) captions.srt
    Retorna dict con paths y texto.
    """
    model = whisper.load_model(model_name)
    result = model.transcribe(audio_path, language=language)
    text = (result.get("text") or "").strip()

    txt_path = f"{out_dir}/transcripcion.txt"
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(text + "\n")

    seg_path = f"{out_dir}/segments.json"
    with open(seg_path, "w", encoding="utf-8") as f:
        json.dump(result.get("segments", []), f, ensure_ascii=False, indent=2)

    srt_path = None
    if generar_srt:
        # whisper CLI genera el .srt en out_dir con el nombre base del audio
        cmd = f'''whisper "{audio_path}" --model {model_name} --language {language} --output_dir "{out_dir}" --output_format srt'''
        subprocess.run(cmd, shell=True, check=False)
        # busca el srt creado
        srts = glob.glob(f"{out_dir}/*.srt")
        if srts:
            srt_path = srts[0]

    return {
        "text": text,
        "transcripcion_txt": txt_path,
        "segments_json": seg_path,
        "srt": srt_path,
    }


##5 — Orquestación:

metadata → si video → transcribe

In [9]:
meta = obtener_metadata(TIKTOK_URL, OUT_DIR)
print("✅ META:")
meta

ERROR: Unsupported URL: https://www.tiktok.com/@ivanrenatoc/photo/7400116644225305862?_r=1&_t=ZS-93fdvANCryS


✅ META:


{'tiktok_url': 'https://vt.tiktok.com/ZSavfXXUo/',
 'final_url': 'https://www.tiktok.com/@ivanrenatoc/photo/7400116644225305862?_r=1&_t=ZS-93fdvANCryS',
 'content_type': 'photo',
 'uploader_name': 'ivanrenatoc',
 'uploader_id': None,
 'uploader_url': None,
 'has_music': False,
 'track': None,
 'artist': None,
 'id': None,
 'title': None,
 'duration_sec': None,
 'view_count': None,
 'like_count': None,
 'comment_count': None,
 'repost_count': None,
 'status': 'error',
 'error': 'ERROR: Unsupported URL: https://www.tiktok.com/@ivanrenatoc/photo/7400116644225305862?_r=1&_t=ZS-93fdvANCryS'}

##6 — Ejecutar transcripción solo si NO es photo

In [10]:
content_type = meta.get("content_type")
if content_type == "photo":
    print("🟡 Es PHOTO. No se transcribe (no hay video estándar para Whisper).")
    trans = None
else:
    print("🟢 Es VIDEO (o video/unknown). Descargando + transcribiendo...")
    video_path = descargar_video(TIKTOK_URL, OUT_DIR)
    audio_path = extraer_audio(video_path, OUT_DIR)
    trans = obtener_texto_whisper(audio_path, OUT_DIR, language=LANG, model_name=WHISPER_MODEL, generar_srt=GENERAR_SRT)
    print("✅ Transcripción lista.")
    print(trans["text"][:1200])

trans


🟡 Es PHOTO. No se transcribe (no hay video estándar para Whisper).


##7 — Mostrar primeros segmentos con timestamps (si existe segments.json)

In [11]:
seg_path = f"{OUT_DIR}/segments.json"
if os.path.exists(seg_path):
    with open(seg_path, "r", encoding="utf-8") as f:
        segs = json.load(f)
    for s in segs[:10]:
        print(f"[{s['start']:.2f}–{s['end']:.2f}] {s['text']}")
else:
    print("No existe segments.json (probablemente era PHOTO o no se transcribió).")

No existe segments.json (probablemente era PHOTO o no se transcribió).


##8 — (Opcional) Descargar outputs

In [ ]:
from google.colab import files

# metadata siempre
files.download(f"{OUT_DIR}/metadata.json")
files.download(f"{OUT_DIR}/video_info_raw.json")

# si hubo transcripción
if os.path.exists(f"{OUT_DIR}/transcripcion.txt"):
    files.download(f"{OUT_DIR}/transcripcion.txt")
if os.path.exists(f"{OUT_DIR}/segments.json"):
    files.download(f"{OUT_DIR}/segments.json")

srts = glob.glob(f"{OUT_DIR}/*.srt")
if srts:
    files.download(srts[0])
